In [1]:
import torch
import torch.autograd as autograd         # computation graph
from torch import Tensor                  # tensor node in the computation graph
import torch.nn as nn                     # neural networks
import torch.optim as optim               # optimizers e.g. gradient descent, ADAM, etc.

import numpy as np
import math
import time
#from pyDOE import lhs         #Latin Hypercube Sampling
import scipy.io
from timeit import default_timer as timer

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as cls

# Device configuration
device = torch.device('cpu')

In [2]:
torch.set_default_dtype(torch.float64)

class SNN(nn.Module):

    def __init__(self, layers1, device):
        super().__init__() #call __init__ from parent class
        self.layers1 = layers1
        self.device = device
        self.activation = nn.Tanh()
        self.loss_function = nn.MSELoss(reduction ='mean')
        self.linears1 = nn.ModuleList([
            nn.Linear(layers1[i], layers1[i+1]).to(dtype=torch.float64) 
            for i in range(len(layers1) - 1)
        ])
        self.M = layers1[-1]
        self.omega = torch.ones((self.M, 1), dtype=torch.float64).to(device)
        self.alpha = 1e-2
        self.a = 0
        self.b = 1
        

        'Xavier Normal Initialization'
        for i1 in range(len(layers1)-1):
            nn.init.xavier_normal_(self.linears1[i1].weight.data, gain=1.0)
            nn.init.zeros_(self.linears1[i1].bias.data)
            
    
        

    'foward pass'
    def forward(self,x):

        if torch.is_tensor(x) != True:
            x = torch.from_numpy(x).to(dtype=torch.float64).to(self.device)
        

        l_x = torch.prod((x - self.a) * (self.b - x), 1)[:,None]
        
        #x = torch.sin( math.pi * x)
        for i in range(len(self.layers1)-2):
            z = self.linears1[i](x)
            x = self.activation(z)
        
        x = self.linears1[-1](x) #+ x_sum

        output = l_x *  x


        return output
                                
    def exact_solution(self,xy):
        x, y = xy[:, 0][:,None], xy[:, 1][:,None]
        temp = torch.sin(math.pi* x)* torch.sin(math.pi* y)
        u_exact = 1 / (1 + 4 * self.alpha * math.pow(math.pi,4)) * temp
        y_exact = 1 / (2 * math.pow(math.pi,2)) * u_exact
        p_exact = - self.alpha * u_exact
        y_d = 1 / (2 * math.pow(math.pi,2)) * temp
        return u_exact, y_exact, p_exact, y_d
    
    
    def loss(self,x):
        g = x
        g.requires_grad = True
        

        fi = self.forward(g)
        
        
        y = fi @ self.omega 
        p = fi @ self.omega 
        u = - 1/self.alpha * p
        
        x1, x2 = x[:,0][:,None],x[:,1][:,None]
        temp = torch.sin(math.pi* x1)* torch.sin(math.pi* x2)
        
        y_x = autograd.grad(y,g,torch.ones([x.shape[0], 1]).to(self.device), retain_graph=True,create_graph=True)[0]
        y_x1 = y_x[:,[0]]
        y_x2 = y_x[:,[1]]
        y_x1_x1x2 = autograd.grad(y_x1,g,torch.ones([x.shape[0], 1]).to(self.device), retain_graph = True, create_graph=True)[0]
        y_x1x1 = y_x1_x1x2[:,[0]]
        y_x2_x1x2 = autograd.grad(y_x2,g,torch.ones([x.shape[0], 1]).to(self.device), retain_graph = True, create_graph=True)[0]
        y_x2x2 = y_x2_x1x2[:,[1]]
        laplace_y = y_x1x1 + y_x2x2

        
                                
        p_x = autograd.grad(p,g,torch.ones([x.shape[0], 1]).to(self.device), retain_graph=True, create_graph=True)[0]
        p_x1 = p_x[:,[0]]
        p_x2 = p_x[:,[1]]
        p_x1_x1x2 = autograd.grad(p_x1,g,torch.ones([x.shape[0], 1]).to(self.device), retain_graph = True, create_graph=True)[0]
        p_x1x1 = p_x1_x1x2[:,[0]]
        p_x2_x1x2 = autograd.grad(p_x2,g,torch.ones([x.shape[0], 1]).to(self.device), retain_graph = True, create_graph=True)[0]
        p_x2x2 = p_x2_x1x2[:,[1]]
        
        laplace_p = p_x1x1 + p_x2x2
        
        y_d = 1 / (2 * math.pow(math.pi,2)) * temp

        
        r_p = - laplace_p   - y + y_d
    
        PDE = - laplace_y  - u

         
        loss_p = self.loss_function(r_p, torch.zeros(r_p.shape).to(self.device))
        loss_PDE = self.loss_function(PDE, torch.zeros(PDE.shape).to(self.device))
    
        return loss_p ,loss_PDE 
                                
                                
                                
    
                                
                                
    def train_model_adam(self, optimizer, x, n_epoch):
        if not torch.is_tensor(x):
            x = torch.from_numpy(x).to(dtype=torch.float64).to(self.device)

        iter_s = 0
        epsilon = 1e-3   # 损失相对下降停止阈值
        gamma = 0.9
        eps = 1e-8
        w_min = 0.3
        w_max = 0.7
        beta = 0.5

        ema_p, ema_PDE = 1.0, 1.0
        initial_loss = None

        while iter_s < n_epoch:
            optimizer.zero_grad()
            loss_p, loss_PDE = self.loss(x)

            # detach 避免影响计算图
            val_p = float(loss_p.detach())
            val_PDE = float(loss_PDE.detach())

            # 更新 EMA
            new_ema_p = gamma * ema_p + (1 - gamma) * val_p
            new_ema_PDE = gamma * ema_PDE + (1 - gamma) * val_PDE

            # 相对改善率 (>=0)
            r1 = max((ema_p - new_ema_p) / (ema_p + eps), 0.0)
            r2 = max((ema_PDE - new_ema_PDE) / (ema_PDE + eps), 0.0)

            # 加入温度参数 alpha，使更新更平滑
            raw_w1 = 1.0 / ((r1 + eps) ** beta)
            raw_w2 = 1.0 / ((r2 + eps) ** beta)

            # 归一化
            w1 = raw_w1 / (raw_w1 + raw_w2)

            # 线性映射到 [w_min, w_max]
            w1 = w_min + (w_max - w_min) * w1
            w2 = 1.0 - w1

            # 组合损失
            loss = w1 * loss_p +  w2 * loss_PDE
            loss1 = loss_p + loss_PDE

            # 初始化停止条件
            if iter_s == 0:
                initial_loss = float(loss1.detach())
                if initial_loss < 1e-12:   # 太小直接停止
                    print("初始损失过小，提前停止")
                    break

            # 相对下降停止条件
            if float(loss1.detach()) / initial_loss < epsilon:
                print("达到损失停止条件:", iter_s,
                      float(loss_p.detach()), float(loss_PDE.detach()),
                      float(loss1.detach()))
                break

            # 反向传播 + 参数更新
            loss.backward()
            optimizer.step()

            # 更新 EMA
            ema_p, ema_PDE = new_ema_p, new_ema_PDE

            # 打印日志
            if iter_s == 0 or iter_s % 100 == 0:
                print(iter_s,
                      float(loss_p.detach()), float(loss_PDE.detach()),
                      float(loss1.detach()))

            iter_s += 1

In [3]:
# === Sampling ===
def sample_domain(n=32):
    x = np.linspace(0, 1, n)
    y = np.linspace(0, 1, n)
    xx, yy = np.meshgrid(x, y)
    xy = np.stack([xx.ravel(), yy.ravel()], axis=-1)
    x_train = torch.tensor(xy, dtype=torch.float64)
    return x_train
def sample_boundary(n=32):
    x = np.linspace(0, 1, n)
    y = np.linspace(0, 1, n)
    points = []
    for xi in x:
        points.append([xi, 0])
        points.append([xi, 1])
    for yi in y:
        points.append([0, yi])
        points.append([1, yi])
    x_b_train = torch.tensor(points, dtype=torch.float64)
    return x_b_train

In [4]:
M = 200
layers1 = np.array([2, 100, 100, 100, 100, M]) 

SNN = SNN(layers1, device)

SNN.to(device)
#print(PINN)
optimizer = torch.optim.Adam(SNN.parameters(), lr=0.001)
x_train = sample_domain(n=32)
start = timer()
SNN.train_model_adam(optimizer, x_train, 1e3)

0 0.44875371937369024 9.039695799441377 9.488449518815067
达到损失停止条件: 61 0.0018382564431444347 0.00725505282540927 0.009093309268553704


In [5]:
phi = SNN.forward(x_train)
Aphi_list = []
for j in range(M):
    phi_j = phi[:, j:j+1]
    grads = autograd.grad(phi_j, x_train, torch.ones_like(phi_j), create_graph=True)[0]
    d2x = autograd.grad(grads[:, 0], x_train, torch.ones_like(grads[:, 0]), create_graph=True)[0][:, 0:1]
    d2y = autograd.grad(grads[:, 1], x_train, torch.ones_like(grads[:, 1]), create_graph=True)[0][:, 1:2]
    Aphi_list.append(-(d2x + d2y))
A_phi = torch.cat(Aphi_list, dim=1)  # [N, M]

In [6]:
u_exact, y_exact, p_exact, y_d = SNN.exact_solution(x_train)
omega_p = torch.ones((M, 1), dtype=torch.float64).to(device)
omega_y = torch.ones((M, 1), dtype=torch.float64).to(device)
x_b_train = sample_boundary(n=32)
phi_x_b = SNN.forward(x_b_train)
u_b_exact, y_b_exact, p_b_exact, y_b_d = SNN.exact_solution(x_b_train)
rhs_y_b = y_b_exact
rhs_p_b = p_b_exact

A_aug = torch.cat([A_phi, phi_x_b], dim=0)  # [N + Nb, M]

In [7]:
epoch = 20
i = 0
alpha = 1e-2

# 避免分母为零
eps = 1e-14

# 保存每次迭代的残差
history = []

while i < epoch:

    # =====================================
    # Step 1: 更新状态系数 omega_y
    # =====================================
    rhs_y = -1 / alpha * phi @ omega_p
    b_y_aug = torch.cat([rhs_y, rhs_y_b], dim=0)

    omega_y = torch.linalg.lstsq(A_aug, b_y_aug).solution

    # =====================================
    # Step 2: 更新伴随系数 omega_p
    # =====================================
    rhs_p = phi @ omega_y - y_d
    b_p_aug = torch.cat([rhs_p, rhs_p_b], dim=0)

    omega_p = torch.linalg.lstsq(A_aug, b_p_aug).solution

    # =====================================
    # Step 3: 计算独立的 PDE 残差
    # =====================================
    # 状态方程残差
    r_y = A_phi @ omega_y + alpha**(-1) * phi @ omega_p

    # 伴随方程残差
    r_p = A_phi @ omega_p - (phi @ omega_y - y_d)

    # =====================================
    # Step 4: 计算独立的边界残差
    # =====================================
    # 状态边界残差
    r_y_bc = phi_x_b @ omega_y - rhs_y_b

    # 伴随边界残差
    r_p_bc = phi_x_b @ omega_p - rhs_p_b

    # =====================================
    # Step 5: 计算绝对 L2 残差和相对 L2 残差
    # =====================================
    # 绝对 L2 残差
    abs_y = torch.linalg.norm(r_y).item()
    abs_p = torch.linalg.norm(r_p).item()

    abs_y_bc = torch.linalg.norm(r_y_bc).item()
    abs_p_bc = torch.linalg.norm(r_p_bc).item()

    # 相对 L2 残差
    rel_y = abs_y / (torch.linalg.norm(rhs_y).item() + eps)
    rel_p = abs_p / (torch.linalg.norm(phi @ omega_y - y_d).item() + eps)

    rel_y_bc = abs_y_bc / (torch.linalg.norm(rhs_y_b).item() + eps)
    rel_p_bc = abs_p_bc / (torch.linalg.norm(rhs_p_b).item() + eps)

    # 保存结果
    history.append({
        "iteration": i + 1,
        "state_abs": abs_y,
        "adjoint_abs": abs_p,
        "state_bc_abs": abs_y_bc,
        "adjoint_bc_abs": abs_p_bc,
        "state_rel": rel_y,
        "adjoint_rel": rel_p,
        "state_bc_rel": rel_y_bc,
        "adjoint_bc_rel": rel_p_bc
    })

    i += 1

    print(
        f"Iteration {i:2d}/{epoch} | "
        f"State PDE: {abs_y:.3e} ({rel_y:.3e}) | "
        f"Adjoint PDE: {abs_p:.3e} ({rel_p:.3e}) | "
        f"State BC: {abs_y_bc:.3e} ({rel_y_bc:.3e}) | "
        f"Adjoint BC: {abs_p_bc:.3e} ({rel_p_bc:.3e})"
    )

end = timer()
print("Consumed time: " + str(end - start) + "s")

Iteration  1/20 | State PDE: 5.486e+00 (2.948e+00) | Adjoint PDE: 5.475e-07 (6.545e-07) | State BC: 7.055e-18 (7.050e-04) | Adjoint BC: 1.393e-18 (1.392e-04)
Iteration  2/20 | State PDE: 1.346e+00 (3.177e-01) | Adjoint PDE: 1.244e-08 (2.180e-08) | State BC: 7.055e-18 (7.050e-04) | Adjoint BC: 1.393e-18 (1.392e-04)
Iteration  3/20 | State PDE: 3.450e-01 (1.193e-01) | Adjoint PDE: 3.954e-09 (6.189e-09) | State BC: 7.055e-18 (7.050e-04) | Adjoint BC: 1.393e-18 (1.392e-04)
Iteration  4/20 | State PDE: 8.854e-02 (2.736e-02) | Adjoint PDE: 3.447e-09 (5.547e-09) | State BC: 7.055e-18 (7.050e-04) | Adjoint BC: 1.393e-18 (1.392e-04)
Iteration  5/20 | State PDE: 2.272e-02 (7.220e-03) | Adjoint PDE: 2.806e-09 (4.484e-09) | State BC: 7.055e-18 (7.050e-04) | Adjoint BC: 1.393e-18 (1.392e-04)
Iteration  6/20 | State PDE: 5.832e-03 (1.840e-03) | Adjoint PDE: 4.950e-09 (7.925e-09) | State BC: 7.055e-18 (7.050e-04) | Adjoint BC: 1.393e-18 (1.392e-04)
Iteration  7/20 | State PDE: 1.497e-03 (4.730e-04) |

In [8]:
def test(x_test):

        if torch.is_tensor(x_test) != True:
            x_test = torch.from_numpy(x_test).to(dtype=torch.float64).to(self.device)
            
        u,y,p,y_d = SNN.exact_solution(x_test)

        phi_pred = SNN.forward(x_test)
        y_pred = phi_pred @ omega_y
        p_pred = phi_pred @ omega_p
        u_pred = -1/SNN.alpha * p_pred
    
        error_y = torch.linalg.norm((y - y_pred),2)/torch.linalg.norm(y,2)
        y_pred = y_pred.cpu().detach().numpy()
        
        error_u = torch.linalg.norm((u - u_pred),2)/torch.linalg.norm(u,2)
        u_pred = u_pred.cpu().detach().numpy()
        
    
        return error_y, y_pred, y,error_u, u_pred, u

In [9]:
x_test = sample_domain(n=100).to(device)
error_y, y_pred, y ,error_u, u_pred, u= test(x_test)
print('Test Error y: {:.2e}'.format(error_y))
print('Test Error u: {:.2e}'.format(error_u))

Test Error y: 1.88e-10
Test Error u: 1.46e-10
